# Complete Pandas Condensed Review Guide
#### The 80/20 Framework - Pandas Mental Model
- **DataFrame** = table  
- **Series** = single column with an index  
- **The index matters** (especially for time series): `set_index`, `reset_index`, sorting, alignment  
- **Typical pandas workflow**: load -> inspect -> clean types -> filter -> create features -> groupby/agg -> join -> export


In [52]:
import pandas as pd
import numpy as np

In [53]:
### 1. Load and set Data
"""
Read:
    pd.read_csv(...) (know parse_dates=, dtype=, na_values=, usecols=)
    pd.read_parquet(...) (fast; common in quant shops)
    pd.read_excel(...)
    pd.read_sql(...) (bonus)
Write:
    df.to_csv(...), df.to_parquet(...), df.to_excel(...)
"""
# Load in data and read col as datetime instead of a string (parse_dates)
trading_data = pd.read_csv("/Users/mburley/Downloads/trading_data.csv", parse_dates = ["date"])
# Set date col as index
trading_data.set_index("date", inplace=True) # inplace = true: modify object directly instead of returning a new one
# Sort rows by date to ensure chronological dates (time series)
trading_data.sort_index(inplace=True)


In [54]:
### 2. Quick inspection
"""
df.head(n), df.tail(n)
df.shape, df.columns, df.index
df.info() (types + missingness)
df.describe() (numeric summary)
df.isna().sum() (missing counts)
df.nunique(), df.value_counts() (categoricals)
"""
print(trading_data.head(5))

           ticker       sector exchange      open      high       low  \
date                                                                    
2025-07-01    SPY          ETF     ARCA  550.8484  556.1887  549.3281   
2025-07-01   AAPL   Technology   NASDAQ  212.7253  212.8582  209.9641   
2025-07-01   MSFT   Technology   NASDAQ  417.8394  423.0622  415.8109   
2025-07-01    GLD  Commodities     ARCA  228.4032  229.7554  228.2981   
2025-07-01    XLE       Energy     ARCA   95.1701   95.3021   94.9157   

               close  adj_close    volume      vwap       bid       ask  \
date                                                                      
2025-07-01  553.9594   554.1763  85941669  551.8862  554.0679  554.1511   
2025-07-01  212.3711   212.3237  80433278  212.0230  212.3395  212.3944   
2025-07-01  420.0211   419.5815  43029677  420.1472  419.7956  419.9147   
2025-07-01  228.7234   228.8927   9115896  229.6661  228.6376  228.6995   
2025-07-01   95.1818    95.1296  11016

In [55]:
### 3. Selecting/filtering (this is the skill)
"""
Columns (1 df statement -> df[...])
    Series: df['col']
    Dataframe: df[['col1', 'col2']]
Rows (2 df statements -> df[df[...]])
    Boolean masks: df[df["x"] > 0]
    Multiple conditions: df[(cond1) & (cond2)] (parentheses matter)
    Membership: df[df["ticker"].isin(list_of_names)]
Indexing
    df.loc[rows, cols] (label-based) 
    df.iloc[rows, cols] (position-based, starts at 0)
"""
# Select closing price col as a series
close_price = trading_data['close']
# Select rows with earnings flag = 1 and event flag = 1, return df
flagged_rows = trading_data[(trading_data['event_flag'] == 1) & (trading_data['earnings_flag'] == 1)]
# Return close and volume prices from 2025-08-01 to 2025-08-04
trading_data.loc['2025-08-01':'2025-08-04', ['close', 'volume']]

## Practice Problem
""" 
Create a DataFrame called `result` that satisfies all of the following:
Keep only rows where:
- `ticker` is either "AAPL" or "MSFT", and
- `volume` is strictly greater than 30_000_000, and
- `spread_bps` is positive. 
Select only the columns:
- `date`, `ticker`, `close`, `volume`, `spread_bps`.
Indexing constraint:
- Use boolean masking for row filtering.
- Use `.loc` for the final row + column selection.
- After filtering, return only the first 10 rows by position using `.iloc`.
Output:  
`result` should be a DataFrame with at most 10 rows and exactly the 5 specified columns.
"""
practice_df = pd.read_csv("/Users/mburley/Downloads/trading_data.csv", parse_dates = ["date"])
filtered_rows = practice_df[practice_df["ticker"].isin(["AAPL", "MSFT"]) & (practice_df['volume'] > 30000000) & (practice_df['spread_bps'] >= 1)]
result = filtered_rows.loc[:, ['date', 'ticker', 'close', 'volume', 'spread_bps']]
result.iloc[:10,:]

,date,ticker,close,volume,spread_bps
3,2025-08-08,MSFT,404.0108,42125036,1.48
6,2025-10-24,MSFT,394.8657,36320193,2.59
7,2025-11-20,MSFT,449.3936,46553785,2.11
11,2025-11-12,MSFT,422.8097,31517688,2.84
12,2025-07-01,MSFT,420.0211,43029677,2.84
16,2025-09-23,AAPL,172.5559,60178603,2.17
26,2025-09-25,MSFT,418.5469,35842437,2.29
27,2025-09-18,AAPL,169.2901,88608124,2.87
28,2025-07-14,AAPL,232.4514,68409122,2.00
29,2025-11-14,AAPL,173.0371,78714089,1.04


In [56]:
### 4. Creating/transforming columns (feature engineering)
"""
    df["new"] = ... (vectorized operations)
    df.assign(new=lambda d: ...) (clean pipelines)
    df.rename(columns={"old":"new"})
    df.drop(columns=[...])
    df.astype(...) (types)
    df.clip(lower=..., upper=...) (winsorize-ish)
    np.where(cond, a, b) (fast branching)
Text
    df["col"].str.lower(), .str.contains(...), .str.extract(...)
Missing data
    df.dropna(), df.fillna(0), df.fillna(method="ffill") (careful in finance)
"""
# Rename notes col to Notes
Notes_df = trading_data.rename(columns = {"notes":"Notes"})
# Convert notes col to uppercase
Notes_df['Notes'].str.upper()

## Practice Problem
""" 
1) Clean text + flag:
- Fill missing values in `notes` with "none", convert to lowercase, and create `has_news` = 1 if `notes` contains "news", else 0.
2) Feature engineering:
- Create `intraday_return = (close - open) / open` and winsorize it to [-0.05, 0.05].
3) Liquidity classification:
- Create `liquidity_flag` = "HIGH" if volume >= 50,000,000, else "LOW" using a vectorized condition.
"""
# Part 1
practice_df['notes'] = practice_df['notes'].fillna("none").str.lower()
practice_df['has_news'] = practice_df['notes'].str.contains('news').astype(int)
# Part 2
open = practice_df['open']
close = practice_df['close']
practice_df['intraday_return'] = (close - open) / open
practice_df['intraday_return'] = practice_df['intraday_return'].clip(-0.05, 0.05)
# Part 3
practice_df['liquidity_flag'] = np.where(practice_df["volume"] >= 50_000_000, "TRUE", "FALSE")
practice_df.head(2)

,date,ticker,sector,exchange,open,high,low,close,adj_close,volume,...,ask,mid,spread_bps,event_flag,earnings_flag,trade_id,notes,has_news,intraday_return,liquidity_flag
0,2025-08-25,SPY,ETF,ARCA,532.9006,537.4994,531.7140,535.8116,535.7799,74968383,...,535.7074,535.6809,0.99,0,0,SPY-20250825-8478,none,0,0.005463,TRUE
1,2025-10-22,XLE,Energy,ARCA,83.8333,84.4665,83.7334,84.2530,84.2607,15701604,...,84.2872,84.2791,1.93,0,0,XLE-20251022-9067,rebalance,0,0.005006,FALSE


In [57]:
### 5. Groupby - Group by KEYS, perform operation on COLUMNS
"""
Gerneral Form: df.groupby(KEYS)[COLUMNS].OPERATION()
KEYS -> column name(s) used to define groups
    Single: df.groupby("ticker")
    Multiple: df.groupby(["ticker", "date"])
COLUMNS -> column(s) you want to compute on
OPERATION -> aggregation/transform/apply
"""
# Calculate the mean close price for each ticker
trading_data.groupby('ticker')['close'].mean()
# Use aggregate to perform multiple operations at once - group each ticker by date
trading_data.groupby(['ticker','date']).agg(
    vol = ('volume','std'), # std dev of the volume col grouped by ticker and date
    vol_sum = ('volume','sum') # sum of each vol grouped by ticker and date
)
## Practice Problem
""" 
Compute a DataFrame with one row per ticker that contains:
    1) avg_daily_return: mean of the daily close-to-close percent return
    2) vol_20d: average of the 20-day rolling standard deviation of daily returns
    3) avg_dollar_volume: mean of (close * volume)
    4) event_day_return: mean daily return on rows where event_flag == 1
Return the result sorted by avg_dollar_volume descending.
"""
# 0) Sort date values
df = pd.read_csv("/Users/mburley/Downloads/trading_data.csv", parse_dates = ["date"])
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['ticker', 'date'])
# 1) Daily returns per ticker
df['daily_rets'] = df.groupby('ticker')['close'].pct_change()
# 2) Calculate 20-day rolling standard deviation of daily returns
df['20day_rolling_std'] = df.groupby('ticker')['daily_rets'].rolling(window=20).std().reset_index(level=0, drop=True)
# 3) Calculate (close * volume)
df['dollar_vol'] = df['close'] * df['volume']
# 4) Calculate daily return on rows where event_flag == 1
df["event_day_rets"] = df["daily_rets"].where(df["event_flag"] == 1)
# Create final df
output_df = df.groupby('ticker').agg(
    avg_daily_return = ('daily_rets', 'mean'),
    vol_20d = ('20day_rolling_std', 'mean'),
    avg_dollar_volume = ('dollar_vol', 'mean'),
    event_day_return = ('event_day_rets', 'mean'))
# Sort as requested
output_df = output_df.sort_values("avg_dollar_volume", ascending=False)
output_df.head()

,avg_daily_return,vol_20d,avg_dollar_volume,event_day_return
ticker,,,,
SPY,0.000178,0.011668,4.568309e+10,-0.009231
AAPL,-0.001404,0.020429,1.677939e+10,0.003524
MSFT,0.001068,0.020152,1.604706e+10,0.003657
GLD,0.002103,0.008748,3.306266e+09,0.001166
XLE,-0.000627,0.017253,1.943878e+09,0.002184


In [ ]:
### 6. Joins / merges (real-world data = stitched together)
'''
pd.merge(left, right, on="key", how="inner")
how = "left" is most common
Multi-key merges: on=["ticker","date"]
Index joins: df.join(other, how="left")
Stack tables: pd.concat([df1, df2], axis=0) (rows) or axis=1 (cols)
'''
prices_df = pd.DataFrame({
    "date": pd.to_datetime(["2025-01-02","2025-01-03","2025-01-02","2025-01-03"]),
    "ticker": ["AAPL","AAPL","MSFT","GLD"],
    "close": [185.2,187.1,410.5,228.3],
    "volume": [72e6,68e6,31e6,12e6]
})

signals_df = pd.DataFrame({
    "date": pd.to_datetime(["2025-01-02","2025-01-03","2025-01-02"]),
    "ticker": ["AAPL","AAPL","MSFT"],
    "signal": [1,-1,1]
})

meta_df = pd.DataFrame({
    "ticker": ["AAPL","MSFT","GLD","XLE"],
    "sector": ["Tech","Tech","Commodities","Energy"],
    "borrow_rate": [0.02,0.015,0.01,0.03]
})
# Inner Join - Only rows where both price AND signal exist
pd.merge(prices_df, signals_df, on = ['date', 'ticker'], how = 'inner')
# Left Join - Keep all prices, attach signals when available, missing signals -> NaN
pd.merge(prices_df, signals_df, on = ['date', 'ticker'], how = 'left')
# Right Join - See signals that don’t have prices (XLE)
pd.merge(prices_df, signals_df, on = ['date', 'ticker'], how = 'right')
## Practice Problem
''' 
You are given three DataFrames: prices_df (daily prices), signals_df (trading signals), and meta_df (security metadata), shown above.
    1. Join the data so all price rows are preserved and metadata is attached, avoiding look-ahead bias.
    2. Compute daily returns, positions, and PnL by ticker.
    3. Aggregate PnL by sector and identify the sector with the highest average daily PnL.
'''
# 1. Use left join to join tables and preserve all data
merged_df = pd.merge(prices_df, signals_df, on=['ticker','date'], how='left')
final_df = pd.merge(merged_df, meta_df, on=['ticker'], how='left')
# 2. Compute daily returns, positions, and PnL
final_df = final_df.sort_values(['ticker', 'date'])
final_df['daily_rets'] = final_df.groupby('ticker')['close'].pct_change()
final_df['position'] = final_df.groupby('ticker')['signal'].shift(1).fillna(0)
final_df['PnL'] = final_df['position'] * final_df['daily_rets']
final_df
# 3. Aggregate PnL by sector and identify the sector with the highest average daily PnL
final_df.groupby('sector').agg( 
    highest_avg_PnL = ('PnL', 'mean')
).sort_values('highest_avg_PnL', ascending=False)

,highest_avg_PnL
sector,
Tech,0.010259
Commodities,NaN


In [85]:
### 7. Time series essentials
'''
Dates
    pd.to_datetime(df["date"])
    df.set_index("date").sort_index()
Returns
    px.pct_change()
    logret = np.log(px).diff()
Rolling
    s.rolling(window).mean(), .std(), .sum()
    s.ewm(span=...).mean() (EMA style)
Resample
    df.resample("1D").last()
    df.resample("M").sum() / .mean()
Shift / lags (avoid lookahead!)
    s.shift(1) (yesterday value)
    signals: signal = (ma_fast > ma_slow).astype(int).shift(1)
'''
trading_data['Rolling_20_mean'] = trading_data['close'].rolling(window = 20).mean()
trading_data.tail()
## Practice Problem
''' 
1. Set `date` as the index, sort by `ticker` and `date`, and compute daily returns per ticker using `adj_close`.
2. For each ticker, compute a 20-day rolling mean and 20-day rolling std of returns, then create a signal = 1 if rolling mean > 0, else 0.
3. Shift the signal by 1 day to avoid lookahead bias and compute strategy returns = signal * daily return.
'''
# Part 0 - Load in data
trade_df = pd.read_csv("/Users/mburley/Downloads/trading_data.csv")
# Part 1
trade_df = trade_df.set_index('date').sort_values(['ticker','date'])
trade_df['daily_rets'] = trade_df.groupby('ticker')['close'].pct_change()
# Part 2
trade_df["ret_mean_20"] = trade_df.groupby('ticker')['daily_rets'].rolling(window = 20).mean().reset_index(level=0, drop=True)
trade_df["ret_std_20"] = trade_df.groupby('ticker')['daily_rets'].rolling(window = 20).std().reset_index(level=0, drop=True)
trade_df["signal"] = (trade_df["ret_mean_20"] > 0).astype(int)
# Part 3
trade_df["signal"] = trade_df.groupby("ticker")["signal"].shift(1).fillna(0)
trade_df['strat_rets'] = trade_df['signal'] * trade_df['daily_rets']
trade_df.tail()

,ticker,sector,exchange,open,high,low,close,adj_close,volume,vwap,...,spread_bps,event_flag,earnings_flag,trade_id,notes,daily_rets,ret_mean_20,ret_std_20,signal,strat_rets
date,,,,,,,,,,,,,,,,,,,,,
2026-01-06,XLE,Energy,ARCA,90.5418,90.9883,89.6129,90.0361,89.9667,21793142,90.3180,...,3.48,1,0,XLE-20260106-3070,NaN,0.000279,0.002881,0.020726,1.0,0.000279
2026-01-07,XLE,Energy,ARCA,88.3858,89.3324,88.2507,88.5145,88.4078,18630912,88.4718,...,3.44,0,0,XLE-20260107-9530,NaN,-0.016900,0.001661,0.021153,1.0,-0.016900
2026-01-08,XLE,Energy,ARCA,83.7848,84.6856,83.1998,84.3194,84.2524,23476613,84.0626,...,3.15,0,0,XLE-20260108-6949,NaN,-0.047394,-0.000756,0.023831,1.0,-0.047394
2026-01-09,XLE,Energy,ARCA,84.3457,84.5496,83.5760,84.0004,83.9516,23870936,NaN,...,2.23,0,0,XLE-20260109-5570,news,-0.003783,-0.001177,0.023805,0.0,-0.000000
2026-01-12,XLE,Energy,ARCA,84.1592,85.8085,83.9073,85.3327,85.3339,18829474,84.7682,...,1.69,0,0,XLE-20260112-1800,NaN,0.015861,-0.001618,0.023379,0.0,0.000000
